In [1]:
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
from tensorflow.keras.optimizers import Adam
import keras
from keras.models import Sequential, Model
from keras.layers import *
from keras.utils import Sequence
from keras.layers import Conv2D, MaxPooling2D
from qkeras import *

from keras.utils import Sequence
from keras.callbacks import CSVLogger
from keras.callbacks import EarlyStopping

import os
import random
from datetime import datetime
import time

import matplotlib.pyplot as plt

pi = 3.14159265359

maxval=1e9
minval=1e-9



2025-06-23 12:54:15.295705: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-06-23 12:54:15.295779: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-06-23 12:54:15.392404: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-23 12:54:15.551253: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-06-23 12:54:28.914869: W tensorflow/compiler/tf2

In [2]:
# os.chdir('SmartPix/data_generator')
os.chdir('/home/das214/SmartPix/mlp_enc_dev')
!pwd

/home/das214/SmartPix/mlp_enc_dev


In [3]:
from OptimizedDataGenerator_v2 import OptimizedDataGenerator
from loss import *
from mlp_encoder_model import *

In [19]:
dataset_base_dir = "/depot/cms/users/das214/datasets/dataset_3sr/dataset_3sr_16x16_50x12P5_parquets/contained"
tfrecords_base_dir = os.path.join(dataset_base_dir, "TFR_files", "2t")

dataset_train_dir = os.path.join(dataset_base_dir, "train")
dataset_test_dir = os.path.join(dataset_base_dir, "test")
tfrecords_dir_train = os.path.join(tfrecords_base_dir, "TFR_train")
tfrecords_dir_val   = os.path.join(tfrecords_base_dir, "TFR_val")

batch_size = 5000
val_batch_size = 5000
train_file_size = len(os.listdir(dataset_train_dir))
val_file_size = len(os.listdir(dataset_test_dir))

In [ ]:
# start_time = time.time()
# validation_generator = OptimizedDataGenerator(
#     dataset_base_dir = dataset_test_dir,
#     file_type = "parquet",
#     data_format = "3D",
#     batch_size = val_batch_size,
#     # optimize_batch_size = True,
#     file_count = val_file_size,
#     to_standardize= True,
#     labels_list = ['x-midplane','y-midplane','cotAlpha','cotBeta'],
#     input_shape = (2,16,16), # (20,13,21),
#     transpose = (0,2,3,1),
#     shuffle = False, 
#     files_from_end=True,

#     tfrecords_dir = tfrecords_dir_val,
#     use_time_stamps = [0,19],
#     max_workers = 2
# )

# print("--- Validation generator %s seconds ---" % (time.time() - start_time))

# # training generator
# start_time = time.time()
# training_generator = OptimizedDataGenerator(
#     dataset_base_dir = dataset_train_dir,
#     file_type = "parquet",
#     data_format = "3D",
#     batch_size = batch_size,
#     # optimize_batch_size = True,
#     file_count = train_file_size,
#     to_standardize= True,
#     labels_list = ['x-midplane','y-midplane','cotAlpha','cotBeta'],
#     input_shape = (2,16,16), # (20,13,21),
#     transpose = (0,2,3,1),
#     shuffle = False, # True 

#     tfrecords_dir = tfrecords_dir_train,
#     use_time_stamps = [0,19],
#     max_workers = 2
# )
# print("--- Training generator %s seconds ---" % (time.time() - start_time))

Processing Files...:   0%|          | 0/20 [00:00<?, ?it/s]

Processing Files...: 100%|██████████| 20/20 [00:04<00:00,  4.20it/s]


Directory /depot/cms/users/das214/datasets/dataset_3sr/dataset_3sr_16x16_50x12P5_parquets/contained/TFR_files/2t/TFR_val does not exist and cannot be removed.


Saving batches as TFRecords: 100%|██████████| 21/21 [00:11<00:00,  1.85it/s]


Metadata saved successfully ast /depot/cms/users/das214/datasets/dataset_3sr/dataset_3sr_16x16_50x12P5_parquets/contained/TFR_files/2t/TFR_val/metadata.json
Loading metadata from /depot/cms/users/das214/datasets/dataset_3sr/dataset_3sr_16x16_50x12P5_parquets/contained/TFR_files/2t/TFR_val/metadata.json
--- Validation generator 16.345064163208008 seconds ---


Processing Files...: 100%|██████████| 80/80 [00:16<00:00,  4.74it/s]


Directory /depot/cms/users/das214/datasets/dataset_3sr/dataset_3sr_16x16_50x12P5_parquets/contained/TFR_files/2t/TFR_train does not exist and cannot be removed.


Saving batches as TFRecords: 100%|██████████| 84/84 [00:32<00:00,  2.57it/s]


Metadata saved successfully ast /depot/cms/users/das214/datasets/dataset_3sr/dataset_3sr_16x16_50x12P5_parquets/contained/TFR_files/2t/TFR_train/metadata.json
Loading metadata from /depot/cms/users/das214/datasets/dataset_3sr/dataset_3sr_16x16_50x12P5_parquets/contained/TFR_files/2t/TFR_train/metadata.json
--- Training generator 49.92697501182556 seconds ---


In [22]:
# Loading pre-generated TFRecords
validation_generator = OptimizedDataGenerator(
    load_from_tfrecords_dir= tfrecords_dir_val,
    shuffle=True,
    seed=42,
    quantize=True,
)

training_generator = OptimizedDataGenerator(
    load_from_tfrecords_dir = tfrecords_dir_train,
    shuffle=True,
    seed=42,
    quantize=True,
)


Loading metadata from /depot/cms/users/das214/datasets/dataset_3sr/dataset_3sr_16x16_50x12P5_parquets/contained/TFR_files/2t/TFR_val/metadata.json
Loading metadata from /depot/cms/users/das214/datasets/dataset_3sr/dataset_3sr_16x16_50x12P5_parquets/contained/TFR_files/2t/TFR_train/metadata.json


In [24]:
model=CreateModel((13,21,2))
model.compile(
    optimizer=tf.keras.optimizers.Nadam(learning_rate=1e-3),
    loss=custom_loss
)

model.summary()

2025-06-23 13:52:31.965114: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory


Model: "smrtpxl_regression"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_pxls/ (InputLayer)    [(None, 13, 21, 2)]          0         []                            
                                                                                                  
 q_activation (QActivation)  (None, 13, 21, 2)            0         ['input_pxls/[0][0]']         
                                                                                                  
 q_activation_1 (QActivatio  (None, 13, 21, 2)            0         ['input_pxls/[0][0]']         
 n)                                                                                               
                                                                                                  
 concatenate (Concatenate)   (None, 26, 21, 2)            0         ['q_activatio

In [10]:
from datetime import datetime

fingerprint = '%08x' % random.randrange(16**8)
timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')
os.makedirs("trained_models", exist_ok=True)
base_dir = f'./trained_models/model-{fingerprint}-checkpoints'
os.makedirs(base_dir, exist_ok=True)  
checkpoint_filepath = base_dir + '/weights.{epoch:02d}-t{loss:.2f}-v{val_loss:.2f}.hdf5'

In [11]:
print(fingerprint)

3f76ef3f


In [12]:
from tensorflow.keras.callbacks import CSVLogger, EarlyStopping, ModelCheckpoint, Callback

early_stopping_patience = 50

class CustomModelCheckpoint(ModelCheckpoint):
    def on_epoch_end(self, epoch, logs=None):
        super().on_epoch_end(epoch, logs)
        checkpoints = [f for f in os.listdir(base_dir) if f.startswith('weights')]
        if len(checkpoints) > 1:
            checkpoints.sort()
            for checkpoint in checkpoints[:-1]:
                os.remove(os.path.join(base_dir, checkpoint))

es = EarlyStopping(patience=early_stopping_patience, restore_best_weights=True)

mcp = CustomModelCheckpoint(
    filepath=checkpoint_filepath,
    save_weights_only=True,
    monitor='val_loss',
    save_best_only=True,
    save_freq='epoch',
    verbose=1
)

csv_logger = CSVLogger(f'{base_dir}/training_log.csv', append=True)

In [ ]:
history = model.fit(
        x=training_generator,
        validation_data=validation_generator,
        callbacks=[es, mcp, csv_logger],
        epochs=1000,
        shuffle=False,
        verbose=1
    )

Epoch 1/1000


2025-06-10 01:39:01.152544: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8907
2025-06-10 01:39:01.598957: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2025-06-10 01:39:02.568350: I tensorflow/core/util/cuda_solvers.cc:179] Creating GpuSolver handles for stream 0x55e6c7d39350
2025-06-10 01:39:04.932586: I external/local_xla/xla/service/service.cc:168] XLA service 0x7ef1196740f0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-06-10 01:39:04.932638: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA A100-PCIE-40GB MIG 1g.5gb, Compute Capability 8.0
2025-06-10 01:39:05.108593: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1749512345.257870 2249739 device_compiler.h:186] Compiled

306/306 [==============================] - ETA: 0s - loss: 18089.2285
Epoch 1: val_loss improved from inf to 3400.99219, saving model to ./trained_models/model-3f76ef3f-checkpoints/weights.01-t18089.23-v3400.99.hdf5
306/306 [==============================] - 40s 103ms/step - loss: 18089.2285 - val_loss: 3400.9922
Epoch 2/1000
306/306 [==============================] - ETA: 0s - loss: -150.4714
Epoch 2: val_loss improved from 3400.99219 to -3909.10254, saving model to ./trained_models/model-3f76ef3f-checkpoints/weights.02-t-150.47-v-3909.10.hdf5
306/306 [==============================] - 29s 94ms/step - loss: -150.4714 - val_loss: -3909.1025
Epoch 3/1000
115/306 [==========>...................] - ETA: 13s - loss: -4569.1069

In [ ]:
'''
Epoch 1/1000
305/305 [==============================] - ETA: 0s - loss: 27173.7227
Epoch 1: val_loss improved from inf to 3957.69629, saving model to ./trained_models/model-e0ef8b33-checkpoints/weights.01-t27173.72-v3957.70.hdf5
305/305 [==============================] - 34s 102ms/step - loss: 27173.7227 - val_loss: 3957.6963
Epoch 2/1000
305/305 [==============================] - ETA: 0s - loss: 2077.7952
Epoch 2: val_loss improved from 3957.69629 to -473.36844, saving model to ./trained_models/model-e0ef8b33-checkpoints/weights.02-t2077.80-v-473.37.hdf5
305/305 [==============================] - 30s 97ms/step - loss: 2077.7952 - val_loss: -473.3684
Epoch 3/1000
305/305 [==============================] - ETA: 0s - loss: -935.6221
Epoch 3: val_loss improved from -473.36844 to -1284.97668, saving model to ./trained_models/model-e0ef8b33-checkpoints/weights.03-t-935.62-v-1284.98.hdf5
305/305 [==============================] - 61s 199ms/step - loss: -935.6221 - val_loss: -1284.9767
Epoch 4/1000
305/305 [==============================] - ETA: 0s - loss: -484.1406
Epoch 4: val_loss improved from -1284.97668 to -1554.20630, saving model to ./trained_models/model-e0ef8b33-checkpoints/weights.04-t-484.14-v-1554.21.hdf5
305/305 [==============================] - 36s 116ms/step - loss: -484.1406 - val_loss: -1554.2063
Epoch 5/1000
305/305 [==============================] - ETA: 0s - loss: -3135.4961
Epoch 5: val_loss improved from -1554.20630 to -2936.62402, saving model to ./trained_models/model-e0ef8b33-checkpoints/weights.05-t-3135.50-v-2936.62.hdf5
305/305 [==============================] - 40s 130ms/step - loss: -3135.4961 - val_loss: -2936.6240
Epoch 6/1000
 38/305 [==>...........................] - ETA: 19s - loss: -3885.3333
'''